In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


## 1. Load model and tokeniser

In [13]:
model=AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",
                                            dtype=torch.float16,
                                            device_map={"": device})
tokenizer=AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

if tokenizer.pad_token==None:
    tokenizer.pad_token=tokenizer.eos_token

model.config.pad_token_id = tokenizer.eos_token_id

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## 2. Load dataset

In [24]:
from datasets import load_dataset

dataset=load_dataset("json",data_files={"train": "train.jsonl", "validation": "valid.jsonl"})

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 117
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 16
    })
})
{'prompt': [{'role': 'user', 'content': 'I finished reading a book.'}], 'completion': [{'role': 'assistant', 'content': 'Finished a book this weekend. Cover to cover. No skimming. 📖🔥\n\nIn a world of 6-second reels and 280-character takes, finishing 312 pages felt like an act of rebellion.\n\nAttention is the new wealth. I just got a little richer.\n\nWhat book is YOUR act of rebellion this month?\n\n#Reading #AttentionSpan #LifelongLearning #DeepWork'}]}


## 3. LoRa config

In [25]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    task_type="CAUSAL_LM",
)

## 4. Training arguments

In [26]:
training_args = SFTConfig(
    output_dir="output/",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=15,
    learning_rate=2e-4,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    completion_only_loss=True,
)

/var/folders/bd/pgfl94xs0jdbrtyb517qg5b00000gn/T/ipykernel_84956/1099802714.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


## 5. Train

In [27]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
)

trainer.train()

/Users/skala/.venvs/global/lib/python3.13/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/skala/.venvs/global/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/117 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

/Users/skala/.venvs/global/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.330364,2.270562,2.261755,0.481609,15435.000000
2,2.067675,2.055883,2.084369,0.514662,30870.000000
3,1.769467,1.981241,1.844064,0.524247,46305.000000
4,1.632903,1.980150,1.668992,0.542062,61740.000000
5,1.416774,2.003599,1.565143,0.539602,77175.000000
6,1.466370,2.043242,1.427633,0.540474,92610.000000
7,1.053913,2.102494,1.371501,0.540552,108045.000000
8,1.060646,2.191575,1.293417,0.539041,123480.000000
9,0.948484,2.249203,1.258383,0.537009,138915.000000
10,0.775668,2.368779,1.157422,0.535576,154350.000000


/Users/skala/.venvs/global/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/skala/.venvs/global/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/skala/.venvs/global/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/skala/.venvs/global/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/skala/.venvs/global/lib/python3.13/site-packages/torc

TrainOutput(global_step=225, training_loss=1.1789583049880135, metrics={'train_runtime': 2777.7016, 'train_samples_per_second': 0.632, 'train_steps_per_second': 0.081, 'total_flos': 3922035693004800.0, 'train_loss': 1.1789583049880135, 'epoch': 15.0})

In [28]:
trainer.save_model("./linkedin-slop-lora-final")
tokenizer.save_pretrained("./linkedin-slop-lora-final")

('./linkedin-slop-lora-final/tokenizer_config.json',
 './linkedin-slop-lora-final/chat_template.jinja',
 './linkedin-slop-lora-final/tokenizer.json')

## 6. Generate

In [6]:
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

device = "mps" if torch.backends.mps.is_available() else "cpu"

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    dtype=torch.float16,
    device_map={"": device},
)
tokenizer = AutoTokenizer.from_pretrained("./linkedin-slop-lora-final")
model = PeftModel.from_pretrained(base_model, "./linkedin-slop-lora-final")
model.eval()

def generate(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False,
    ).to(device)
    with torch.no_grad():
        output = model.generate(
            inputs=input_ids,
            attention_mask=torch.ones_like(input_ids),
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)



Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [8]:
prompts = [
    "I worked until 2am.",
    "My dog puked on the couch and I had to clean it.",
    "The salad I ate at lunch gave me food poisoning.",
]

for prompt in prompts:
    with model.disable_adapter():
        base_out = generate(prompt)
    tuned_out = generate(prompt)

    print(f"PROMPT: {prompt}")
    print("-" * 15)
    print(f"* BASE:   {base_out}")
    print("-" * 5)
    print(f"* TUNED:  {tuned_out}")
    print("=" * 60)

PROMPT: I worked until 2am.
---------------
* BASE:   That sounds like a long and tiring night. Working until 2am can be exhausting, both physically and mentally. Hopefully, you got everything done that you needed to, and you're looking forward to a good rest and some well-deserved relaxation. Do you have any fun plans for the day ahead, or are you just looking forward to a quiet day at home?
-----
* TUNED:  Worked until 2am tonight. Not because I was 'fired up' or 'in flow.' Because sometimes the only way to get it done is to be the only one who can see it's not done. 🛠️📆

That 2am epiphany? Worth it.

That 8-hour gap between 'almost done' and 'finally done'? Worth every restless night.

Finish what nobody else will. Even if nobody's watching except yourself.

2am. My favorite hour.

#LateNightWins #Discipline #FinishWhatYouStart #NightOwlConfessions
PROMPT: My dog puked on the couch and I had to clean it.
---------------
* BASE:   That's definitely not the most fun experience. Cleani